# Primitive move

# Getting Charges Data

In [41]:
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langsmith import Client

import urllib.error
import urllib.request

from langchain.tools import tool
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from IPython.display import Markdown, display
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool

from pydantic import BaseModel, Field
import ch_charges as ch_ch
import ch_filing_history as ch_fh
import json
load_dotenv() # looks for a .env file in the current or parent directions

api_key = os.getenv('ANTHROPIC_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')

In [68]:
number = "10812571"
since = '2015-01-01'
CATS = ch_fh.SIGNAL_CATEGORIES

filings = ch_fh.get_filing_history(number)
kept    = ch_fh.filter_filings(filings, since=since, categories = CATS)
records = [ch_fh.summarise(f) for f in kept]

charges = ch_ch.get_charges(number)
records_ch = [ch_ch.summarise(c) for c in charges]

In [69]:
records_ch

[{'charge_code': '108125710002',
  'created_on': '2017-07-10',
  'delivered_on': '2017-07-20',
  'satisfied_on': None,
  'status': 'outstanding',
  'classification': 'A registered charge',
  'persons_entitled': ['Interbay Funding Limited'],
  'charge_id': 'fPgLKs6NTqFzZ0_vqa-SMrhO72A'},
 {'charge_code': '108125710001',
  'created_on': '2017-07-10',
  'delivered_on': '2017-07-20',
  'satisfied_on': None,
  'status': 'outstanding',
  'classification': 'A registered charge',
  'persons_entitled': ['Interbay Funding Limited'],
  'charge_id': 'WbYPpoOpnetYtgzrMQrX-zR59RE'}]

## Identify Structured Output

In [ ]:
class FilingSummary(BaseModel):
    """What one company's filing history says about its recent activity."""
    headline: str = Field(
        description = "One senctence, 20 words maximum. No lists, no semicolons."    
    )                # one line, human-readable
    key_events: list[str] = Field(
        description="The 2-3 most significant filings. Each must begin with its date.",
        max_length=3,
    )           # the 2-3 filings that matter, each dated
    has_existing_charges: bool = Field(
        description="True if any mortgage-category filing appears in the data provided."
    )        # any category == "mortgage"?
    charge_data: str | None = Field(
        description="If has_existing_charges is true, explain the existing charges data"
    )
    latest_accounts_date: str | None = Field(
        description="The 'made up to' date of the most recent accounts filing, as "
                    "YYYY-MM-DD. This is the accounting period end, NOT the date the "
                    "accounts were filed. Null if no accounts filing is present."
    )
    filing_gaps: str | None = Field(
        description="Late or missing filings worth flagging to a credit analyst. "
                    "Null if the filing record is unremarkable."
    )           # late or missing filings worth flagging

## Specify System prompt

In [ ]:
ROLE = """You are a credit analyst reading a UK company's Companies House filing history AND registered chareges, if any.
Summarise what the filings and the charges show about the company's recent activity
and anything relevant to its borrowing position."""

SCOPE = (f"You are seeing only filings in these categories: {', '.join(sorted(CATS))},"
         f"dated {since} or later. Everything else was removed before you saw it. "
         f"Do not draw any conclusion from the absence of other filing types.")

SYSTEM = f"{ROLE}\n\n{SCOPE}"

## Initiate Model

In [72]:
model = init_chat_model('claude-sonnet-5')
summariser = model.with_structured_output(FilingSummary)
result = summariser.invoke([
    {"role" : "system", "content" : SYSTEM},
    {"role" : "user", "content" : f"Filing history : \n{json.dumps(records, indent=2)}, Charges : \n{json.dumps(records_ch, indent = 2)}"}
])

## Result

In [73]:
from IPython.display import Markdown, display

def show(obj):
    d = obj.model_dump()
    lines = [f"### {d.pop('headline', '')}"]
    for k, v in d.items():
        label = k.replace('_', ' ').title()
        if isinstance(v, list):
            lines.append(f"**{label}**")
            lines += [f"- {i}" for i in v] or ["- —"]
        else:
            lines.append(f"**{label}:** {v if v is not None else '—'}")
    display(Markdown("\n\n".join(lines)))

show(result)

### Small company with two outstanding charges to Interbay Funding and a late 2024 accounts filing.

**Key Events**

- 2017-06-09: Company incorporated

- 2017-07-20: Two mortgages created in favour of Interbay Funding Limited (charge nos. 108125710001/002), both still outstanding

- 2025-12-17: Micro-entity accounts filed for year to 2024-06-30, filed well over a year after period end

**Has Existing Charges:** True

**Charge Data:** Two outstanding registered charges, both created 10 July 2017 and delivered 20 July 2017, with Interbay Funding Limited as the person entitled. Neither charge has been satisfied, indicating ongoing secured lending against company assets, likely property given Interbay's specialism in property finance.

**Latest Accounts Date:** 2024-06-30

**Filing Gaps:** Accounts for year end 2024-06-30 were filed on 2025-12-17, roughly 18 months after period end, well beyond the normal 9-month deadline for private companies, suggesting administrative or compliance issues. Earlier accounts (2018-2020) were paper filed, a minor inefficiency but not a compliance concern.